In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)
np.random.seed(42)

In [2]:
df = pd.read_csv('dataset_invernadero.csv')

# Número de niveles de cada actuador
SALIDAS = {'Resistencia': 3, 'Ventilador': 3, 'Persianas': 3, 'Luces': 2, 'Humidificador': 2}

In [3]:
def preparar_entradas(df):
    X = pd.DataFrame({
        'T':   df['Temperatura'] / 40.0,     # 0–40 °C
        'L':   df['Luminosidad'] / 1000.0,   # 0–1000
        'Hum': df['Humedad'] / 100.0,        # 0–100 %
    })
    hora = pd.get_dummies(df['Hora'].astype(int)).reindex(columns=[0, 1, 2], fill_value=0)
    X[['Manana', 'Dia', 'Noche']] = hora.values
    return X.values.astype('float32')

X = preparar_entradas(df)
Y = {k: df[k].values for k in SALIDAS}

idx = np.arange(len(df))
idx_train, idx_test = train_test_split(idx, test_size=0.2, random_state=42, stratify=df['Ventilador'])

X_train, X_test = X[idx_train], X[idx_test]
y_train = {k: v[idx_train] for k, v in Y.items()}
y_test  = {k: v[idx_test]  for k, v in Y.items()}
print(X_train.shape, X_test.shape)

(800, 6) (200, 6)


In [4]:
entrada = keras.Input(shape=(X.shape[1],))
x = layers.Dense(64, activation='relu')(entrada)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dense(32, activation='relu')(x)

salidas = {}
for nombre, n in SALIDAS.items():
    if n == 3:
        salidas[nombre] = layers.Dense(3, activation='softmax', name=nombre)(x)
    else:
        salidas[nombre] = layers.Dense(1, activation='sigmoid', name=nombre)(x)

model = keras.Model(entrada, salidas)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.003),
    loss={k: ('sparse_categorical_crossentropy' if n == 3 else 'binary_crossentropy') for k, n in SALIDAS.items()},
    metrics={k: 'accuracy' for k in SALIDAS},
)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 6)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │        448 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      4,160 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │      2,080 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Humidificador       │ (None, 1)         │         33 │ dense_2[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Luces (Dense)       │ (None, 1)         │         33 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Persianas (Dense)   │ (None, 3)         │         99 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Resistencia (Dense) │ (None, 3)         │         99 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Ventilador (Dense)  │ (None, 3)         │         99 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 7,051 (27.54 KB)

 Trainable params: 7,051 (27.54 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=60, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=20, min_lr=1e-5),
]

print('Iniciando entrenamiento...')
history = model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=1500, batch_size=32,
    callbacks=callbacks,
    verbose=0,   # cambia a 1 si quieres ver cada época
)
print(f"Entrenamiento terminado en {len(history.history['loss'])} épocas")

Iniciando entrenamiento...
Entrenamiento terminado en 203 épocas


In [6]:
def predecir(model, X):
    p = model.predict(X, verbose=0)
    return {k: (p[k].argmax(axis=1) if n == 3 else (p[k].ravel() > 0.5).astype(int))
            for k, n in SALIDAS.items()}

pred = predecir(model, X_test)
aciertos = np.column_stack([pred[k] == y_test[k] for k in SALIDAS])

print('Precisión por actuador:')
for k, acc in zip(SALIDAS, aciertos.mean(axis=0)):
    print(f'  {k:<14} {acc*100:6.2f}%')
print(f'\nCasos con las 5 salidas correctas: {aciertos.all(axis=1).mean()*100:.2f}%')

Precisión por actuador:
  Resistencia     97.00%
  Ventilador      99.00%
  Persianas       99.00%
  Luces          100.00%
  Humidificador   99.00%

Casos con las 5 salidas correctas: 95.00%


In [7]:
# Dónde se equivoca (filas = real, columnas = predicho)
for k, n in SALIDAS.items():
    print(k)
    print(confusion_matrix(y_test[k], pred[k], labels=list(range(n))), '\n')

Resistencia
[[149   0   2]
 [  0  23   2]
 [  0   2  22]] 

Ventilador
[[54  0  0]
 [ 1 48  0]
 [ 0  1 96]] 

Persianas
[[151   1   0]
 [  0  30   1]
 [  0   0  17]] 

Luces
[[183   0]
 [  0  17]] 

Humidificador
[[119   1]
 [  1  79]] 



In [8]:
nueva = pd.DataFrame([{'Temperatura': 18, 'Hora': 1, 'Luminosidad': 250, 'Humedad': 80}])
print(predecir(model, preparar_entradas(nueva)))
# Esperado por las reglas: Resistencia=1, Ventilador=1, Persianas=2, Luces=1, Humidificador=0

{'Resistencia': array([1]), 'Ventilador': array([1]), 'Persianas': array([2]), 'Luces': array([1]), 'Humidificador': array([0])}
